<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-12-production-deploy/lesson-12.4-streamlit-frontend/practice/GCP_Capstone_12.4_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 12.4 — Streamlit Frontend

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup (run first)

This lesson builds a multi-file Streamlit app. In Colab we **author the files** exactly the way the lesson notebook does (writing each module from a string constant) and validate them — the actual Streamlit process runs on Cloud Run, not inside Colab. Run this cell once to install pinned deps, authenticate with ADC, and set the environment placeholders the modules read.

Conventions: SDK `google-genai` (`genai.Client(enterprise=True, ...)`), models `gemini-3.6-flash` / `gemini-3.1-pro-preview` / `gemini-3.1-flash-lite`, region `us-central1` (course) / `asia-south1` (India speech), `USD_INR = 85`.

In [ ]:
%%bash
pip install -q \
  streamlit==1.55.0 \
  openai==1.78.1 \
  litellm==1.77.3 \
  google-genai==2.20.0 \
  google-cloud-storage==2.19.0 \
  google-cloud-documentai==3.5.0 \
  google-cloud-speech==2.27.0 \
  google-cloud-texttospeech==2.27.0 \
  'PyJWT[crypto]==2.10.1' \
  tiktoken==0.9.0 \
  tenacity==9.0.0

In [ ]:
# Application Default Credentials (Colab) - never API keys
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Colab ADC ready')
except ImportError:
    print('Not on Colab - assuming gcloud ADC is configured')

import os

# --- Project / region placeholders (swap for your worked-example ids) ---
os.environ.setdefault('GOOGLE_CLOUD_PROJECT', 'documind-ai-YOUR-ID')
os.environ.setdefault('GOOGLE_CLOUD_LOCATION', 'us-central1')   # course region
os.environ.setdefault('SPEECH_REGION', 'asia-south1')           # India speech
os.environ.setdefault('UPLOAD_BUCKET', 'documind-ai-YOUR-ID-uploads')
os.environ.setdefault('TTS_CACHE_BUCKET', 'documind-ai-YOUR-ID-tts-cache')
os.environ.setdefault('LAYOUT_PROCESSOR',
    'projects/documind-ai-YOUR-ID/locations/us/processors/YOUR_PROCESSOR_ID')
os.environ.setdefault('LITELLM_BASE_URL', 'https://litellm.internal')
os.environ.setdefault('LITELLM_MASTER_KEY', 'sk-REPLACE-ME')
os.environ.setdefault('AUTH_MODE', 'oidc')  # 'iap' in prod

USD_INR = 85

# Sanity: the unified google-genai Vertex client (enterprise=True, NOT vertexai=True)
from google import genai
from google.genai import types
client = genai.Client(
    enterprise=True,
    project=os.environ['GOOGLE_CLOUD_PROJECT'],
    location=os.environ['GOOGLE_CLOUD_LOCATION'],
)
print('genai Vertex client ready:', os.environ['GOOGLE_CLOUD_PROJECT'],
      '/', os.environ['GOOGLE_CLOUD_LOCATION'])

In [ ]:
# requirements.txt - pinned April 2026 (lesson Cell 1). The Docker image installs these.
REQUIREMENTS = '''\
streamlit==1.55.0
Authlib==1.6.6
openai==1.78.1
litellm==1.77.3
google-cloud-aiplatform==1.95.0
google-cloud-discoveryengine==0.13.11
google-cloud-storage==2.19.0
google-cloud-documentai==3.5.0
google-cloud-firestore==2.20.0
google-cloud-bigquery[pandas]==3.29.0
google-cloud-secret-manager==2.22.0
google-cloud-speech==2.27.0
google-cloud-texttospeech==2.27.0
google-cloud-dlp==3.27.0
google-genai==2.20.0
langchain-text-splitters==0.3.0
streamlit-pdf-viewer==0.0.28
streamlit-mic-recorder==0.0.8
streamlit-cookies-manager==0.2.0
PyJWT[crypto]==2.10.1
tiktoken==0.9.0
tenacity==9.0.0
fpdf2==2.8.3
plotly==5.24.1
pandas==2.2.3
'''
with open('requirements.txt', 'w') as f:
    f.write(REQUIREMENTS)
print('requirements.txt written')

## Exercise 1: Streamlit Dockerfile

**Difficulty:** Easy

Dockerfile with python:3.12-slim + tini + healthcheck on /_stcore/health. CMD with XSRF disabled.

1. Base on `python:3.12-slim`, install `tini` + `curl` and create a non-root `app` user.
2. Copy and `pip install -r requirements.txt`, then copy the app (chown to `app`) and drop to the non-root user.
3. Set the Streamlit server env (headless, port 8080, CORS off, **XSRF off** so the load balancer doesn't trip 403s).
4. Add a `HEALTHCHECK` that curls `/_stcore/health`, use `tini` as the entrypoint, and pass the XSRF/CORS flags in `CMD` too.

**Expected:** Image builds, healthcheck passes, no XSRF 403s behind the LB.

In [ ]:
# Lifted from lesson Cell 7. tini as PID 1, non-root uid 10001, health on /_stcore/health,
# XSRF + CORS disabled (required behind a Cloud Run / LB proxy that terminates TLS).
DOCKERFILE = '''
# syntax=docker/dockerfile:1.7
FROM python:3.12-slim

RUN apt-get update && apt-get install -y --no-install-recommends \\
      tini curl ca-certificates build-essential \\
    && rm -rf /var/lib/apt/lists/*

RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip \\
 && pip install --no-cache-dir -r requirements.txt

COPY --chown=app:app . .
USER app

ENV PORT=8080 \\
    STREAMLIT_SERVER_PORT=8080 \\
    STREAMLIT_SERVER_ADDRESS=0.0.0.0 \\
    STREAMLIT_SERVER_HEADLESS=true \\
    STREAMLIT_SERVER_ENABLE_CORS=false \\
    STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION=false \\
    STREAMLIT_BROWSER_GATHER_USAGE_STATS=false \\
    PYTHONUNBUFFERED=1

EXPOSE 8080
HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \\
    CMD curl -fsS http://localhost:8080/_stcore/health || exit 1

ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["streamlit","run","app.py", \\
     "--server.port=8080","--server.address=0.0.0.0", \\
     "--server.headless=true", \\
     "--server.enableCORS=false","--server.enableXsrfProtection=false"]
'''
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)
print('Dockerfile written')

## Exercise 2: Chat skeleton

**Difficulty:** Easy

`st.chat_message` + `st.chat_input` loop with `session_state` message history. Emoji avatars.

1. Seed `st.session_state.messages` with a system prompt on first run.
2. On every rerun, replay the history with `st.chat_message`, skipping the system turn.
3. Read new input with `st.chat_input`, append the user turn, and echo a placeholder assistant turn.
4. Give user and assistant distinct emoji avatars so bubbles are visually distinct.

In [ ]:
# Minimal skeleton grounded in chat_page() from lesson Cell 3 (streaming/cost stripped out;
# those come back in Exercise 4). Persistence + avatars are the whole point here.
CHAT_SKELETON_PY = '''
import streamlit as st

def chat_page(user):
    st.title("\U0001f4ac DocuMind Chat")

    # History lives in session_state so it survives Streamlit reruns
    if "messages" not in st.session_state:
        st.session_state.messages = [{"role": "system", "content": "You are DocuMind."}]

    # Replay history every rerun (skip the system turn)
    for m in st.session_state.messages:
        if m["role"] == "system":
            continue
        with st.chat_message(m["role"], avatar="\U0001f9d1" if m["role"] == "user" else "\U0001f916"):
            st.markdown(m["content"])

    if prompt := st.chat_input("Ask DocuMind...", max_chars=4000):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user", avatar="\U0001f9d1"):
            st.markdown(prompt)
        # Placeholder reply - Exercise 4 swaps this for a real streamed completion
        reply = f"(echo) You said: {prompt}"
        with st.chat_message("assistant", avatar="\U0001f916"):
            st.markdown(reply)
        st.session_state.messages.append({"role": "assistant", "content": reply})
'''
with open('chat_skeleton.py', 'w') as f:
    f.write(CHAT_SKELETON_PY)
print('chat_skeleton.py written')

## Exercise 3: Cloud Run deploy

**Difficulty:** Easy

`gcloud run deploy` with `--session-affinity` + `--cpu-boost` + `--timeout=3600`. Verify WebSocket works via browser DevTools.

1. Deploy the image with session affinity, CPU boost, gen2 execution env, and a 3600s timeout (long-lived WebSockets).
2. Restrict ingress to internal + load balancing, keep `--no-allow-unauthenticated`, and wire the runtime service account + secrets.
3. Grant the service account self-impersonation (`serviceAccountTokenCreator`) so it can sign V4 URLs.
4. Turn on IAP 1-click and grant a user `iap.httpsResourceAccessor`, then confirm the `ws://` upgrade in DevTools.

**Expected:** DevTools Network tab shows the `ws://` upgrade, no 403s, session cookie present.

In [ ]:
%%bash
# Lifted from lesson Cell 7. Set PROJECT / GIT_SHA before running.
# --session-affinity pins a viewer to one instance (Streamlit is stateful);
# --cpu-boost speeds cold starts; --timeout=3600 keeps the WebSocket alive.
export PROJECT="documind-ai-YOUR-ID"
export GIT_SHA="$(git rev-parse --short HEAD 2>/dev/null || echo dev)"

gcloud run deploy documind-ui \
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/ui:$GIT_SHA \
  --region=us-central1 --platform=managed \
  --no-allow-unauthenticated \
  --ingress=internal-and-cloud-load-balancing \
  --memory=2Gi --cpu=2 --concurrency=80 --timeout=3600 \
  --min-instances=1 --max-instances=10 \
  --cpu-boost --session-affinity --execution-environment=gen2 \
  --service-account=documind-ui-sa@$PROJECT.iam.gserviceaccount.com \
  --set-env-vars="AUTH_MODE=iap,LITELLM_BASE_URL=https://litellm.internal,SPEECH_REGION=asia-south1,GOOGLE_CLOUD_PROJECT=documind-ai-YOUR-ID,GOOGLE_CLOUD_LOCATION=us-central1,UPLOAD_BUCKET=documind-ai-YOUR-ID-uploads,LAYOUT_PROCESSOR=projects/documind-ai-YOUR-ID/locations/us/processors/YOUR_LAYOUT_PROC,TTS_CACHE_BUCKET=documind-ai-YOUR-ID-tts-cache,VECTOR_INDEX_ENDPOINT=YOUR_INDEX_ENDPOINT,DEPLOYED_INDEX_ID=documind_deployed,CHUNKS_COLLECTION=chunks,AUDIT_COLLECTION=audit_events,OBSERVABILITY_DATASET=documind_observability,RAG_MODEL=gemini-3.6-flash,ADMIN_EMAILS=admin@documind.ai,ADMIN_DOMAINS=documind.ai,IAP_AUDIENCE=YOUR_IAP_AUDIENCE" \
  --set-secrets="LITELLM_MASTER_KEY=litellm-master-key:latest,COOKIE_SECRET=cookie-secret:latest" \
  --vpc-connector=projects/$PROJECT/locations/us-central1/connectors/documind-vpc \
  --vpc-egress=private-ranges-only

# Grant self-impersonation for V4 signed URLs
gcloud iam service-accounts add-iam-policy-binding \
  documind-ui-sa@$PROJECT.iam.gserviceaccount.com \
  --member="serviceAccount:documind-ui-sa@$PROJECT.iam.gserviceaccount.com" \
  --role="roles/iam.serviceAccountTokenCreator"

# Enable IAP 1-click
gcloud beta run services update documind-ui --region=us-central1 --iap

# Grant a user access through IAP
gcloud beta iap web add-iam-policy-binding \
  --resource-type=cloud-run --service=documind-ui --region=us-central1 \
  --member="user:alice@documind.ai" \
  --role=roles/iap.httpsResourceAccessor

## Exercise 4: LiteLLM streaming + stop button

**Difficulty:** Medium

`stream_completion()` generator with a `stop_requested` flag. `stream_options` for authoritative usage. Tenacity retry on `APIConnectionError`.

1. Build an OpenAI-compatible client pointed at the LiteLLM gateway (`base_url` + master key), `max_retries=0` so Tenacity owns retries.
2. Open the stream with `stream_options={"include_usage": True}` and wrap the call in a Tenacity retry that fires only on `APIConnectionError`.
3. Yield token deltas; break early when `st.session_state.stop_requested` is set.
4. On the final usage-only chunk, compute cost from the per-model `PRICING` table and accumulate session cost + token counts.

**Expected:** Stop button interrupts cleanly, usage matches LiteLLM `/spend`, retries on a network blip.

In [ ]:
# Lifted from lesson Cell 3 (chat.py). Full streaming module: retry-wrapped stream open,
# stop flag, authoritative usage accounting, and the cost sidebar (USD + INR at 85).
CHAT_PY = '''
import os, time
import streamlit as st
from openai import OpenAI, APIError, APIConnectionError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import tiktoken

ENC = tiktoken.get_encoding("cl100k_base")

PRICING = {
    "gemma-on-cloudrun":      {"in": 0.0,  "out": 0.0},
    "gemini-3.1-flash-lite":  {"in": 0.25, "out": 1.50},
    "gemini-3.6-flash":       {"in": 1.50, "out": 7.50},
    "gemini-3.1-pro-preview": {"in": 2.00, "out": 12.00},
}

@st.cache_resource
def get_openai():
    return OpenAI(api_key=os.environ["LITELLM_MASTER_KEY"],
                  base_url=os.environ["LITELLM_BASE_URL"],
                  timeout=120.0, max_retries=0)

@retry(reraise=True, stop=stop_after_attempt(3),
       wait=wait_exponential(multiplier=0.5, max=4),
       retry=retry_if_exception_type(APIConnectionError))
def _open_stream(model, messages, temp, top_p, max_tok):
    return get_openai().chat.completions.create(
        model=model, messages=messages, temperature=temp,
        top_p=top_p, max_tokens=max_tok, stream=True,
        stream_options={"include_usage": True})

def stream_completion(messages):
    ss = st.session_state
    ss.stop_requested = False
    ss.is_streaming = True
    try:
        stream = _open_stream(ss.model, messages, ss.temperature, ss.top_p, ss.max_tokens)
        for chunk in stream:
            if ss.stop_requested:
                yield "\\n\\n_(stopped)_"; break
            if getattr(chunk, "usage", None) and not chunk.choices:
                cost = (chunk.usage.prompt_tokens * PRICING[ss.model]["in"]
                        + chunk.usage.completion_tokens * PRICING[ss.model]["out"]) / 1_000_000
                ss.session_cost_usd = ss.get("session_cost_usd", 0) + cost
                ss.last_turn_cost = cost
                ss.tokens_in = ss.get("tokens_in", 0) + chunk.usage.prompt_tokens
                ss.tokens_out = ss.get("tokens_out", 0) + chunk.usage.completion_tokens
                break
            if not chunk.choices: continue
            text = getattr(chunk.choices[0].delta, "content", None) or ""
            if text: yield text
    finally:
        ss.is_streaming = False

def chat_page(user):
    with st.sidebar:
        st.session_state.model = st.selectbox("Model",
            ["gemma-on-cloudrun", "gemini-3.6-flash", "gemini-3.1-pro-preview"])
        st.session_state.temperature = st.slider("Temperature", 0.0, 2.0, 0.7)
        st.session_state.top_p = 0.95
        st.session_state.max_tokens = 1024
        st.divider()
        st.subheader("\U0001f4b0 Cost this session")
        ss = st.session_state
        st.metric("USD", f"${ss.get(\'session_cost_usd\', 0):.4f}",
                  delta=f"+${ss.get(\'last_turn_cost\', 0):.4f} last turn")
        st.metric("INR", f"₹{ss.get(\'session_cost_usd\', 0) * 85:.2f}")
        st.metric("Tokens", f"{ss.get(\'tokens_in\', 0)} in / {ss.get(\'tokens_out\', 0)} out")

    st.title("\U0001f4ac DocuMind Chat")
    if "messages" not in st.session_state:
        st.session_state.messages = [{"role":"system","content":"You are DocuMind."}]

    for m in st.session_state.messages:
        if m["role"] == "system": continue
        with st.chat_message(m["role"], avatar="\U0001f9d1" if m["role"]=="user" else "\U0001f916"):
            st.markdown(m["content"])

    col1, col2 = st.columns([5, 1])
    with col2:
        if st.button("⏹ Stop", disabled=not st.session_state.get("is_streaming", False)):
            st.session_state.stop_requested = True

    if prompt := st.chat_input("Ask DocuMind...", max_chars=4000):
        st.session_state.messages.append({"role":"user","content":prompt})
        with st.chat_message("user", avatar="\U0001f9d1"):
            st.markdown(prompt)
        with st.chat_message("assistant", avatar="\U0001f916"):
            reply = st.write_stream(stream_completion(st.session_state.messages))
        st.session_state.messages.append({"role":"assistant","content":reply})
'''
with open('chat.py', 'w') as f:
    f.write(CHAT_PY)
print('chat.py written')

## Exercise 5: Document upload pipeline

**Difficulty:** Medium

GCS resumable upload + Doc AI Layout Parser v1.5 + `text-embedding-005` + Vector Search upsert with a per-user restrict.

1. Stream the uploaded file to GCS with an 8 MB chunk size (resumable) under a per-user prefix.
2. Run the Doc AI **Layout Parser v1.5** processor with chunking (chunk_size 500, ancestor headings) and pull `chunked_document.chunks`.
3. Embed the chunks in batches of 5 with `text-embedding-005` (`RETRIEVAL_DOCUMENT`, 768-dim).
4. Upsert to Vector Search restricted to the user's `sub` (Module 11 pattern).

**Expected:** 50 MB PDF uploads, parses, embeds, upserts. Queryable in Vector Search within 10s.

In [ ]:
# Lifted from lesson Cell 4 (documents.py). Uses the unified google-genai client
# (enterprise=True) for embeddings, Doc AI Layout Parser v1.5 for chunking.
DOCS_PY = '''
import os
import streamlit as st
from google.cloud import storage, documentai_v1 as docai
from google import genai
from google.genai import types
from google.cloud import aiplatform

_storage = storage.Client()
BUCKET = _storage.bucket(os.environ["UPLOAD_BUCKET"])
_docai = docai.DocumentProcessorServiceClient(
    client_options={"api_endpoint": "us-documentai.googleapis.com"})
_genai = genai.Client(enterprise=True, project=os.environ.get("GOOGLE_CLOUD_PROJECT"), location=os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1"))
LAYOUT_PROCESSOR = os.environ["LAYOUT_PROCESSOR"]

def extract_chunks(doc, source_uri):
    chunks = []
    for i, c in enumerate(doc.chunked_document.chunks):
        chunks.append({
            "id": f"{source_uri}#{i}",
            "text": c.content,
            "page_start": c.page_span.page_start if c.page_span else None,
            "source_uri": source_uri,
        })
    return chunks

def parse_layout(gcs_uri):
    req = docai.ProcessRequest(
        name=f"{LAYOUT_PROCESSOR}/processorVersions/pretrained-layout-parser-v1.5-2025-08-25",
        gcs_document=docai.GcsDocument(gcs_uri=gcs_uri, mime_type="application/pdf"),
        process_options=docai.ProcessOptions(
            layout_config=docai.ProcessOptions.LayoutConfig(
                chunking_config=docai.ProcessOptions.LayoutConfig.ChunkingConfig(
                    chunk_size=500, include_ancestor_headings=True))))
    return _docai.process_document(request=req).document

def embed_batch(chunks):
    for i in range(0, len(chunks), 5):
        batch = chunks[i:i+5]
        resp = _genai.models.embed_content(
            model="text-embedding-005",
            contents=[c["text"] for c in batch],
            config=types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT", output_dimensionality=768))
        for c, e in zip(batch, resp.embeddings):
            c["embedding"] = e.values
    return chunks

def documents_page(user):
    st.title("\U0001f4c4 Documents")
    files = st.file_uploader("Upload documents for indexing",
                             type=["pdf", "docx", "txt", "md"],
                             accept_multiple_files=True,
                             max_upload_size=200)

    if files and st.button("Index documents"):
        with st.status("Processing...", expanded=True) as status:
            for f in files:
                st.write(f"\U0001f4e4 Uploading {f.name}")
                blob = BUCKET.blob(f"users/{user[\'sub\']}/{f.file_id}/{f.name}")
                blob.chunk_size = 8 * 1024 * 1024
                blob.upload_from_file(f, content_type=f.type, timeout=300)
                gcs_uri = f"gs://{BUCKET.name}/{blob.name}"

                st.write(f"\U0001f50d Parsing layout ({f.name})")
                doc = parse_layout(gcs_uri)
                chunks = extract_chunks(doc, gcs_uri)

                st.write(f"\U0001f4c8 Embedding {len(chunks)} chunks")
                chunks = embed_batch(chunks)

                st.write(f"\U0001f4be Upserting to Vector Search")
                # upsert_datapoints(chunks, user["sub"])  # Module 11 pattern - restrict on user sub

            status.update(label="Done!", state="complete")
'''
with open('documents.py', 'w') as f:
    f.write(DOCS_PY)
print('documents.py written')

## Exercise 6: Citation pills + jump-to-page

**Difficulty:** Medium

`[N]` regex → HTML pills with hover tooltips. Expandable sources with V4 signed URLs + `#page=N` anchors.

1. Regex-match `[N]` and `[N,M]` markers in the model answer and replace each with a styled HTML `<span>` pill carrying a hover tooltip of the source snippet.
2. Render the rewritten answer with `unsafe_allow_html=True`.
3. Below the answer, show an expander of source cards.
4. Give each card a V4 signed URL (`SIGNED_URL_EXPIRY`) with a `#page=N` fragment so "Jump" opens the PDF at the cited page.

**Expected:** `[1]` `[2,3]` render as clickable pills, expander shows source cards, Jump opens the PDF at the page.

In [ ]:
# citations.py - the lesson lists this module (Cell 8) but ships no code for it, so this is
# written fresh to the practice-lab spec: [N] regex -> pills, V4 signed URLs, #page=N anchors.
CITATIONS_PY = '''
import os, re
import datetime
import streamlit as st
from google.cloud import storage

_storage = storage.Client()
SIGNED_URL_EXPIRY = datetime.timedelta(minutes=15)
_CITE = re.compile(r"\\[(\\d+(?:\\s*,\\s*\\d+)*)\\]")

def signed_url(gcs_uri, page=None):
    # gcs_uri: gs://bucket/path -> V4 signed URL; SA needs serviceAccountTokenCreator on itself
    _, _, rest = gcs_uri.partition("gs://")
    bucket_name, _, blob_name = rest.partition("/")
    blob = _storage.bucket(bucket_name).blob(blob_name)
    url = blob.generate_signed_url(version="v4", expiration=SIGNED_URL_EXPIRY, method="GET")
    return f"{url}#page={page}" if page else url

def render_with_citations(answer, sources):
    # sources: list of {"text":..., "source_uri":..., "page_start":...}, index N -> sources[N-1]
    def _pill(match):
        nums = [int(n) for n in match.group(1).replace(" ", "").split(",")]
        spans = []
        for n in nums:
            src = sources[n - 1] if 0 < n <= len(sources) else None
            tip = (src["text"][:120] + "...") if src else "unknown source"
            tip = tip.replace(\'"\', \'&quot;\')
            spans.append(
                f\'<span title="{tip}" style="background:#ccfbf1;color:#065f46;\'
                f\'border-radius:6px;padding:1px 7px;margin:0 2px;font-size:12px;\'
                f\'font-weight:600;cursor:help;">[{n}]</span>\')
        return "".join(spans)
    html = _CITE.sub(_pill, answer)
    st.markdown(html, unsafe_allow_html=True)

    with st.expander(f"\U0001f4da Sources ({len(sources)})"):
        for i, src in enumerate(sources, 1):
            page = src.get("page_start")
            st.markdown(f"**[{i}]** {src[\'text\'][:200]}...")
            try:
                url = signed_url(src["source_uri"], page=page)
                label = f"Jump to page {page}" if page else "Open source"
                st.link_button(label, url)
            except Exception as e:
                st.caption(f"(signed URL unavailable: {e})")
'''
with open('citations.py', 'w') as f:
    f.write(CITATIONS_PY)
print('citations.py written (fresh - lesson notebook ships no citations.py body)')

In [ ]:
# rag.py - the lesson lists this module but ships no body; written fresh here so
# chat.py's `from rag import answer_query` resolves. embed -> ANN (tenant filter) -> rerank -> answer.
RAG_PY = '''
"""rag.py - DocuMind retrieval: query embedding -> ANN search -> rerank -> grounded answer.

Backend (provisioned in Modules 4/11/12):
  - Vertex AI Vector Search index endpoint   (env VECTOR_INDEX_ENDPOINT, DEPLOYED_INDEX_ID)
  - Firestore `chunks` collection keyed by datapoint id -> {text, source_uri, page_start}
  - LiteLLM gateway for generation            (env LITELLM_BASE_URL, LITELLM_MASTER_KEY)
  - genai embeddings (text-embedding-005, served regionally)
Tenant isolation: every datapoint carries a `tenant_id` restrict = user["sub"].
The answer's [N] markers line up with the returned `sources`, so
citations.render_with_citations(result["answer"], result["sources"]) renders pills.
"""
import os
import streamlit as st
from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud import aiplatform
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import Namespace
from openai import OpenAI

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
REGION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")   # embeddings + vector search are regional
EMBED_MODEL = "text-embedding-005"
EMBED_DIMS = 768
INDEX_ENDPOINT = os.environ.get("VECTOR_INDEX_ENDPOINT", "")     # projects/.../locations/.../indexEndpoints/123
DEPLOYED_INDEX_ID = os.environ.get("DEPLOYED_INDEX_ID", "")
CHUNKS_COLLECTION = os.environ.get("CHUNKS_COLLECTION", "chunks")

SYSTEM = (
    "You are DocuMind, a document assistant. Answer ONLY from the numbered sources. "
    "Cite every claim inline with [N] using the source's number. If the sources do "
    "not contain the answer, say so plainly. Be concise.")


@st.cache_resource
def _embed_client():
    # Embeddings are served from the regional endpoint (NOT the global generation endpoint).
    return genai.Client(enterprise=True, project=PROJECT, location=REGION)


@st.cache_resource
def _index_endpoint():
    aiplatform.init(project=PROJECT, location=REGION)
    return aiplatform.MatchingEngineIndexEndpoint(INDEX_ENDPOINT)


@st.cache_resource
def _fs():
    return firestore.Client(project=PROJECT, database="(default)")


@st.cache_resource
def _llm():
    # Generation goes through the LiteLLM gateway (same virtual key as chat.py).
    return OpenAI(api_key=os.environ.get("LITELLM_MASTER_KEY", "sk-none"),
                  base_url=os.environ.get("LITELLM_BASE_URL", "http://localhost:4000"),
                  timeout=120.0, max_retries=1)


def embed_query(query):
    resp = _embed_client().models.embed_content(
        model=EMBED_MODEL, contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY",
                                        output_dimensionality=EMBED_DIMS))
    return resp.embeddings[0].values


def retrieve(query, tenant_id, top_k=8):
    """Embed the query and ANN-search this tenant's shard; hydrate chunk text from Firestore."""
    if not INDEX_ENDPOINT or not DEPLOYED_INDEX_ID:
        return []
    vec = embed_query(query)
    restricts = [Namespace(name="tenant_id", allow_tokens=[tenant_id])]
    resp = _index_endpoint().find_neighbors(
        deployed_index_id=DEPLOYED_INDEX_ID,
        queries=[vec], num_neighbors=top_k, filter=restricts)
    neighbors = resp[0] if resp else []
    fs = _fs()
    out = []
    for n in neighbors:
        snap = fs.collection(CHUNKS_COLLECTION).document(n.id).get()
        data = snap.to_dict() if snap.exists else {}
        out.append({
            "id": n.id,
            "text": data.get("text", ""),
            "source_uri": data.get("source_uri", ""),
            "page_start": data.get("page_start"),
            "distance": getattr(n, "distance", None),
        })
    return out


def rerank(query, chunks, top_n=4):
    """Re-order ANN candidates with the Vertex AI Ranking API; fall back to ANN distance order."""
    if not chunks:
        return []
    try:
        from google.cloud import discoveryengine_v1 as de
        client = de.RankServiceClient()
        ranking_config = client.ranking_config_path(
            project=PROJECT, location="global", ranking_config="default_ranking_config")
        records = [de.RankingRecord(id=str(i), content=c["text"][:2000])
                   for i, c in enumerate(chunks) if c["text"]]
        if not records:
            return chunks[:top_n]
        ranked = client.rank(request=de.RankRequest(
            ranking_config=ranking_config, model="semantic-ranker-default@latest",
            query=query, records=records, top_n=top_n))
        return [chunks[int(r.id)] for r in ranked.records]
    except Exception:
        # Ranking API not enabled / unavailable -> keep ANN order (nearest first).
        return sorted(chunks, key=lambda c: (c["distance"] is None, c["distance"] or 0.0))[:top_n]


def answer_query(query, tenant_id, top_k=8, top_n=4, model=None):
    """Full RAG turn: embed -> ANN -> rerank -> grounded generation with [N] citations.

    Returns {answer, sources, model, num_retrieved}; `sources` is exactly what
    citations.render_with_citations() expects: [{text, source_uri, page_start}, ...].
    """
    model = model or os.environ.get("RAG_MODEL", "gemini-3.6-flash")
    candidates = retrieve(query, tenant_id, top_k=top_k)
    sources = rerank(query, candidates, top_n=top_n)
    if not sources:
        return {"answer": "I couldn't find anything in your indexed documents for that. "
                          "Upload documents on the Documents page first.",
                "sources": [], "model": model, "num_retrieved": 0}
    context = "\\n\\n".join(
        f"[{i}] (source: {s.get('source_uri', '?')}, page {s.get('page_start', '?')})\\n{s['text']}"
        for i, s in enumerate(sources, 1))
    resp = _llm().chat.completions.create(
        model=model, temperature=0.1, max_tokens=1024,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"Sources:\\n{context}\\n\\nQuestion: {query}"}])
    return {"answer": resp.choices[0].message.content or "",
            "sources": sources, "model": model, "num_retrieved": len(sources)}'''
with open('rag.py', 'w') as f:
    f.write(RAG_PY)
print('rag.py written')

## Exercise 7: Voice I/O with Hinglish

**Difficulty:** Challenge

`streamlit-mic-recorder` + Chirp 3 STT `language_codes=["auto"]` + Chirp 3: HD streaming TTS + SHA-256 GCS cache. Test with "Mujhe Q4 report chahiye".

1. Transcribe recorded audio with Chirp 3 STT via the Speech v2 regional endpoint (`asia-south1`), multi-language codes for Hinglish code-switching, auto punctuation.
2. Fall back through `chirp_2` / `long` models if the primary recognizer errors.
3. Synthesize replies with Chirp 3: HD streaming TTS (OGG/Opus) using a named speaker.
4. Key a GCS cache by SHA-256 of `voice|rate|encoding|text`; serve the blob on a cache hit, synthesize + store on a miss.

**Expected:** Transcribes Hinglish correctly. Plays a Chirp 3: HD Hindi voice. Cache hit < 100ms.

In [ ]:
# Lifted from lesson Cell 5 (voice.py). Speech v2 regional endpoint (asia-south1),
# model fallback chain, and a SHA-256 GCS cache that also zeroes repeat TTS cost.
VOICE_PY = '''
import os, hashlib
import streamlit as st
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech as cs
from google.api_core.client_options import ClientOptions
from google.cloud import texttospeech as tts, storage

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
REGION = os.environ.get("SPEECH_REGION", "asia-south1")
TTS_CACHE_BUCKET = storage.Client().bucket(os.environ["TTS_CACHE_BUCKET"])

_speech = SpeechClient(client_options=ClientOptions(
    api_endpoint=f"{REGION}-speech.googleapis.com"))
_tts = tts.TextToSpeechClient()

def transcribe(audio_bytes, language_codes=("hi-IN", "en-IN"), model="chirp_3"):
    cfg = cs.RecognitionConfig(
        auto_decoding_config=cs.AutoDetectDecodingConfig(),
        language_codes=list(language_codes),
        model=model,
        features=cs.RecognitionFeatures(enable_automatic_punctuation=True))
    req = cs.RecognizeRequest(
        recognizer=f"projects/{PROJECT}/locations/{REGION}/recognizers/_",
        config=cfg, content=audio_bytes)
    resp = None
    try: resp = _speech.recognize(request=req)
    except Exception:
        for m in ("chirp_2", "long"):
            cfg.model = m; req.config = cfg
            try: resp = _speech.recognize(request=req); break
            except Exception: continue
    if resp is None:
        return ""   # all STT attempts failed; degrade gracefully
    return " ".join(r.alternatives[0].transcript for r in resp.results
                    if r.alternatives).strip()

def cached_tts(text, voice="en-IN-Chirp3-HD-Kore", lang="en-IN", rate=1.0):
    cache_key = hashlib.sha256(f"{voice}|{rate}|ogg|{text}".encode()).hexdigest()
    blob = TTS_CACHE_BUCKET.blob(f"tts/{cache_key}.ogg")
    if blob.exists():
        return blob.download_as_bytes()
    # Cache miss - synthesize
    cfg = tts.StreamingSynthesizeConfig(
        voice=tts.VoiceSelectionParams(language_code=lang, name=voice),
        streaming_audio_config=tts.StreamingAudioConfig(
            audio_encoding=tts.AudioEncoding.OGG_OPUS, speaking_rate=rate))
    def gen():
        yield tts.StreamingSynthesizeRequest(streaming_config=cfg)
        yield tts.StreamingSynthesizeRequest(
            input=tts.StreamingSynthesisInput(text=text))
    audio = b"".join(r.audio_content for r in _tts.streaming_synthesize(gen()))
    blob.upload_from_string(audio, content_type="audio/ogg")
    return audio
'''
with open('voice.py', 'w') as f:
    f.write(VOICE_PY)
print('voice.py written')
print('Chirp 3 STT: hi-IN + en-IN for Hinglish ("Mujhe Q4 report chahiye")')
print('Chirp 3: HD TTS + SHA-256 GCS cache -> 1.2s to ~80ms and $0 on repeats')

## Exercise 8: Full UI — auth + cost tracker + admin

**Difficulty:** Challenge

IAP JWT verification + LiteLLM virtual-key mint + sidebar cost widget + admin dashboard for usage/tenants/audit.

1. In IAP mode, read `x-goog-iap-jwt-assertion` and **verify** it (ES256, JWKS, audience, issuer) — never trust the plain email header. In dev, use native `st.login()`.
2. Gate the app behind `login_gate()` and gate the admin nav entry with `is_admin()` (admin emails / domains).
3. Wire `app.py` to route Chat / Documents / Admin, mint a LiteLLM virtual key on first login, and keep the sidebar cost widget (from Exercise 4).
4. Add an admin dashboard with usage / tenants / audit tabs, hidden from non-admins.

**Expected:** A forged email header is rejected, a LiteLLM key is minted on first login, the admin tab is hidden from non-admins.

In [ ]:
# auth.py - lifted from lesson Cell 6. Verifies the IAP JWT (ES256/JWKS/aud/iss);
# falls back to native st.login() in dev. is_admin() gates the admin nav entry.
AUTH_PY = '''
import os
import streamlit as st
import jwt
from jwt import PyJWKClient

IAP_AUDIENCE = os.getenv("IAP_AUDIENCE")
ADMIN_EMAILS = set(os.getenv("ADMIN_EMAILS", "").split(","))
ADMIN_DOMAINS = set(os.getenv("ADMIN_DOMAINS", "").split(","))
_JWKS = PyJWKClient("https://www.gstatic.com/iap/verify/public_key-jwk")

def _verify_iap_jwt(token, audience):
    key = _JWKS.get_signing_key_from_jwt(token).key
    return jwt.decode(token, key, algorithms=["ES256"],
                      audience=audience, issuer="https://cloud.google.com/iap")

def current_user():
    if os.getenv("AUTH_MODE") == "iap":
        h = st.context.headers or {}
        token = h.get("x-goog-iap-jwt-assertion")
        if not token:
            return {"is_logged_in": False}
        try:
            claims = _verify_iap_jwt(token, IAP_AUDIENCE)
            return {"email": claims["email"], "sub": claims["sub"],
                    "is_logged_in": True}
        except Exception as e:
            st.error(f"IAP JWT verification failed: {e}")
            return {"is_logged_in": False}
    if st.user.is_logged_in:
        return {"email": st.user.email, "sub": st.user.sub,
                "name": getattr(st.user, "name", ""), "is_logged_in": True}
    return {"is_logged_in": False}

def login_gate():
    u = current_user()
    if not u["is_logged_in"]:
        st.title("\U0001f510 DocuMind - Sign in required")
        if os.getenv("AUTH_MODE") != "iap":
            st.button("Log in with Google", on_click=st.login)
        else:
            st.error("IAP authentication missing. Contact admin.")
        st.stop()
    return u

def is_admin(user):
    email = user.get("email", "").lower()
    domain = email.split("@")[-1] if "@" in email else ""
    return email in ADMIN_EMAILS or domain in ADMIN_DOMAINS
'''
with open('auth.py', 'w') as f:
    f.write(AUTH_PY)
print('auth.py written')

In [ ]:
# app.py - lifted from lesson Cell 2. Auth gate + nav; admin entry only for admins.
APP_PY = '''
import os
import streamlit as st
from auth import login_gate, is_admin
from chat import chat_page
from documents import documents_page
from admin_dashboard import admin_page

st.set_page_config(page_title="DocuMind", page_icon="\U0001f4c4", layout="wide",
                   initial_sidebar_state="expanded")

user = login_gate()

pages = ["Chat", "Documents", "Admin"] if is_admin(user) else ["Chat", "Documents"]
with st.sidebar:
    st.markdown(f"### \U0001f464 {user.get(\'email\')}")
    if st.button("Sign out"):
        st.logout() if os.getenv("AUTH_MODE") != "iap" else st.markdown("Close tab to sign out of IAP")
    st.divider()
    page = st.radio("Navigation", pages, label_visibility="collapsed")

if page == "Chat":
    chat_page(user)
elif page == "Documents":
    documents_page(user)
elif page == "Admin":
    admin_page(user)
'''
with open('app.py', 'w') as f:
    f.write(APP_PY)
print('app.py written')

In [ ]:
# admin_dashboard.py - fresh skeleton. The lesson lists this module (Cell 8) but ships no body;
# written to the practice spec: server-side is_admin() re-check + usage/tenants/audit tabs.
ADMIN_PY = '''
import streamlit as st
from auth import is_admin

def admin_page(user):
    # Defense in depth: hiding the nav entry is not access control - re-check here
    if not is_admin(user):
        st.error("403 - Admins only")
        st.stop()

    st.title("\U0001f6e1 Admin Dashboard")
    tab_usage, tab_tenants, tab_audit = st.tabs(["Usage", "Tenants", "Audit Log"])

    with tab_usage:
        st.subheader("Spend by model (from LiteLLM /spend)")
        # rows = litellm_client.get_spend()  # wire to LiteLLM admin API
        st.info("Query LiteLLM /spend and BigQuery usage export here.")

    with tab_tenants:
        st.subheader("Tenants / virtual keys")
        st.info("List minted virtual keys, budgets, and per-tenant caps.")

    with tab_audit:
        st.subheader("Audit log")
        st.info("Stream Cloud Audit Logs / DLP findings for RESTRICTED replies.")
'''
with open('admin_dashboard.py', 'w') as f:
    f.write(ADMIN_PY)
print('admin_dashboard.py written (fresh - lesson notebook ships no admin_dashboard.py body)')

In [ ]:
# LiteLLM virtual-key mint on first login (fresh glue - referenced in the lab spec,
# no code in the lesson notebook). Call this from login_gate() on first sign-in.
import os
from openai import OpenAI  # LiteLLM speaks the OpenAI protocol; key mint is a plain POST
import requests

def mint_virtual_key(user_sub, max_budget_usd=10.0):
    """Mint a per-user LiteLLM virtual key with a hard budget. Idempotent by key_alias."""
    resp = requests.post(
        f"{os.environ['LITELLM_BASE_URL']}/key/generate",
        headers={"Authorization": f"Bearer {os.environ['LITELLM_MASTER_KEY']}"},
        json={
            "key_alias": f"user-{user_sub}",
            "max_budget": max_budget_usd,
            "models": ["gemini-3.6-flash", "gemini-3.1-pro-preview", "gemma-on-cloudrun"],
            "duration": "30d",
        },
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()["key"]

print('mint_virtual_key() defined - call on first login, store the returned key in session_state')